In [7]:
!pip install -q yfinance

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from models.wmt_model import WMTTradingModel
from models.nvda_model import NVDATradingModel
from models.mpc_model import MPCTradingModel
from models.xom_model import XOMTradingModel

from utils import ForecastingMetrics, TradingMetrics, PortfolioEvaluator
from backtest import Backtest
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize

The best models for each stock are:
- WMT: VAR(0)
    - Provided lower error, higher robustness, and simpler logic
- MPC: ARIMA/ARIMA-GARCH
    - Captured both trend and shock-adjustment behavior
    - Compared to the other potential best models (Ridge and Naive) it had the smallest errors
- NVDA: VAR(0)
    - This stock is very volatile 
    - More event-driven rather than dependent on historical data so 
        - enforcing lag-based structure increased forecast error 
    - The mean forecast (VAR(0)) minimized cumulative prediction error better than models trying to impose structure where none existed
- XOM: VAR(1)
    - Provided the best balance between low forecast error, directional accuracy, and alignment

In [30]:
def _markowitz_from_forecasts(forecasts: np.ndarray,
                              objective: str = "gmir") -> tuple[np.ndarray, float]:
    """
    Compute Markowitz weights from a (T, N) matrix of forecasted returns.

    Parameters
    ----------
    forecasts : np.ndarray
        Matrix of forecasted returns with shape (horizon, n_assets).
    objective : {"gmir", "gmv"}
        "gmir" - maximize mean / volatility (information ratio).
        "gmv"  - global minimum variance.

    Returns
    -------
    weights : np.ndarray
        Optimal long-only weights summing to 1, shape (n_assets,).
    obj_value : float
        Information ratio (for gmir) or portfolio variance (for gmv).
    """
    forecasts = np.asarray(forecasts, dtype=float)
    mu = forecasts.mean(axis=0)              # expected return per asset
    Sigma = np.cov(forecasts, rowvar=False)  # covariance matrix

    n = len(mu)
    w0 = np.ones(n) / n
    bounds = [(0.0, 1.0)] * n
    cons = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)

    if objective == "gmir":
        def neg_ir(w: np.ndarray) -> float:
            r = float(w @ mu)
            vol = float(np.sqrt(w @ Sigma @ w))
            if vol == 0:
                return 1e6
            return -r / vol

        res = minimize(neg_ir, w0, bounds=bounds, constraints=cons)
        w_opt = res.x
        ir = -neg_ir(w_opt)
        return w_opt, ir

    elif objective == "gmv":
        def var_obj(w: np.ndarray) -> float:
            return float(w @ Sigma @ w)

        res = minimize(var_obj, w0, bounds=bounds, constraints=cons)
        w_opt = res.x
        return w_opt, var_obj(w_opt)

    else:
        raise ValueError("objective must be 'gmir' or 'gmv'")


def forecast_and_markowitz(as_of_date: str,
                           horizon: int = 60,
                           lookback_days: int = 252,
                           objective: str = "gmir"
                           ) -> tuple[pd.DataFrame, pd.Series]:
    """
    For any date after Jan 2025, re-optimize the portfolio using
    forecasted returns from the best model per stock.

    Parameters
    ----------
    as_of_date : str
        Date string "YYYY-MM-DD". Use any date >= "2025-01-01".
        All data strictly before this date is used for training.
    horizon : int, default=60
        Number of future trading days to forecast.
    lookback_days : int, default=252
        Number of past trading days used for model estimation.
    objective : {"gmir", "gmv"}, default="gmir"
        Markowitz objective: max information ratio or min variance.

    Returns
    -------
    forecast_df : pd.DataFrame
        Shape (horizon, 4) with columns ["WMT", "NVDA", "MPC", "XOM"]
        containing forecasted daily returns.
    weights : pd.Series
        Markowitz optimal weights indexed by ticker, summing to 1.
    """
    tickers = ["WMT", "NVDA", "MPC", "XOM"]
    model_map = {
        "WMT": WMTTradingModel,          # VAR(0) mean model
        "NVDA": NVDATradingModel,        # VAR(0) mean model
        "MPC": MPCTradingModel,          # ARIMA(1,1,1)
        "XOM": XOMTradingModel,          # AR(1) via ARIMA(1,0,0)
    }

    end = pd.to_datetime(as_of_date)
    start = end - pd.tseries.offsets.BDay(int(lookback_days * 1.5))

    forecasts_list: list[np.ndarray] = []

    for t in tickers:
        # 1) download prices up to as_of_date
        data = yf.download(t, start=start, end=end)
        closes = data["Close"].dropna().tail(lookback_days)

        # 2) compute log-returns
        log_ret = np.log(closes).diff().dropna().values

        # 3) fit best model and produce 60-day forecast
        model_cls = model_map[t]
        model = model_cls()
        model.fit(log_ret)

        dummy_X = np.zeros(horizon)
        fc = np.asarray(model.predict(dummy_X), dtype=float)
        forecasts_list.append(fc)

    # 4) stack forecasts into (horizon, n_assets) matrix
    forecast_matrix = np.column_stack(forecasts_list)
    forecast_df = pd.DataFrame(forecast_matrix, columns=tickers)

    # 5) compute Markowitz weights using forecasted returns
    weights_arr, _ = _markowitz_from_forecasts(forecast_matrix, objective=objective)
    weights = pd.Series(weights_arr, index=tickers, name="weight")

    return forecast_df, weights

In [31]:
forecast_df, weights = forecast_and_markowitz("2025-02-01")

print("First 5 of the 60-day forecasts:")
display(forecast_df.head())

print("\nMarkowitz weights from forecasted returns:")
print(weights)

/tmp/ipykernel_8283/305941040.py:99: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_8283/305941040.py:99: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_8283/305941040.py:99: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_8283/305941040.py:99: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(t, start=start, end=end)
[*********************100%***********************]  1 of 1 completed

First 5 of the 60-day forecasts:


,WMT,NVDA,MPC,XOM
0,0.002349,0.002665,-0.000453,0.000648
1,0.002349,0.002665,-0.000430,0.000280
2,0.002349,0.002665,-0.000430,0.000285
3,0.002349,0.002665,-0.000430,0.000285
4,0.002349,0.002665,-0.000430,0.000285



Markowitz weights from forecasted returns:
WMT     7.342635e-09
NVDA    1.000000e+00
MPC     0.000000e+00
XOM     0.000000e+00
Name: weight, dtype: float64
